# CAP 5610 Midterm — Task 1
## Logistic Regression on the Adult Census Income Dataset

The goal is to predict whether an individual's annual income is greater
than $50,000. I will compare logistic regression with and without class
weighting, establish a majority-class baseline, and evaluate the models
using accuracy, F1-score, and ROC-AUC.

## 1. Import Required Libraries

This section imports the libraries used for data handling, preprocessing,
model training, and evaluation.

- `numpy` and `pandas` are used for numerical operations and data handling.
- `matplotlib` is used for plots.
- `scikit-learn` provides preprocessing tools, logistic regression,
  pipelines, and evaluation metrics.

In [19]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

#For Missing Values
from sklearn.impute import SimpleImputer
#For Data Pre-Processing
from sklearn.compose import ColumnTransformer

#For Logistic Regression
from sklearn.linear_model import LogisticRegression

#Stastical Scores Required for this task
from sklearn.metrics import(accuracy_score,
f1_score,
roc_auc_score,
confusion_matrix,
classification_report
)

#Importing pipeline to combine preprocessing and modelling into one workflow
from sklearn.pipeline import Pipeline

#Importing One-Hot encoder to convert categorical variables to numerical values and Standard Scaler to standardize continuous
#numerical variables
from sklearn.preprocessing import StandardScaler,OneHotEncoder

print("All Imports Successful")

All Imports Successful


## 2. Load the Adult Census Training and Test Data

-> We have been provided separate training and test datasets.

-> The training set will be used to fit the preprocessing steps and the
Logistic Regression model. The test set will only be used after training
to evaluate how well the model generalizes to unseen observations.

In [20]:
# Paths are relative to this notebook.
# The notebook is inside the "notebooks" folder, so "../data/"
# moves up one level and then enters the "data" folder.

train_path = "../data/adult_train_.csv"
test_path = "../data/adult_test_.csv"

# Load the supplied CSV files into pandas DataFrames.
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

## 3. Initial Inspection of the Dataset

Before preprocessing or training a model, we first inspect the size,
column names, data types, and a few example rows.

This helps verify that the files were loaded correctly and also helps
identify which variables are numerical and which are categorical.

In [21]:
print("Training Shape:", train_df.shape)
print("Testinging Shape:", test_df.shape)

train_df.head()

Training Shape: (26048, 14)
Testinging Shape: (6513, 14)


,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,label->50K
0,33,Local-gov,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,50,United-States,1
1,36,Private,Assoc-voc,11,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,1887,50,United-States,1
2,58,Self-emp-not-inc,9th,5,Separated,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,0
3,21,Private,Assoc-voc,11,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,46,United-States,0
4,27,Private,Some-college,10,Divorced,Other-service,Not-in-family,White,Male,0,0,40,United-States,0


In [22]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26048 entries, 0 to 26047
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             26048 non-null  int64
 1   workclass       26048 non-null  str  
 2   education       26048 non-null  str  
 3   education_num   26048 non-null  int64
 4   marital_status  26048 non-null  str  
 5   occupation      26048 non-null  str  
 6   relationship    26048 non-null  str  
 7   race            26048 non-null  str  
 8   sex             26048 non-null  str  
 9   capital_gain    26048 non-null  int64
 10  capital_loss    26048 non-null  int64
 11  hours_per_week  26048 non-null  int64
 12  native_country  26048 non-null  str  
 13  label->50K      26048 non-null  int64
dtypes: int64(6), str(8)
memory usage: 2.8 MB


## 4. Inspect Columns and Data Types

The Adult dataset contains both continuous and categorical variables.
Before the preprocessing pipeline is created, the column names and data
types are inspected so that an appropriate transformation can be applied
to each feature.

The target variable is `label->50K`.

In [23]:
#Display all columns as they appear
print("Columns:")
print(train_df.columns.tolist())

#Display data type for each column
print("Data Type:")
print(train_df.dtypes)

Columns:
['age', 'workclass', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'label->50K']
Data Type:
age               int64
workclass           str
education           str
education_num     int64
marital_status      str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country      str
label->50K        int64
dtype: object


## 5. Check for Missing Values

Missing values must be handled before any processing may begin. Following steps are needed:
1. The dataset is first checked for standard missing values such as `NaN`.
2. However, the Adult dataset may also use the string `"?"` to represent
missing categorical values. Therefore, both forms are checked.

In [24]:
print("Standard Missing Values in the Train Dataset:")
print(train_df.isna().sum())

#Doing the same for test set as well
print("\nStandard missing values in Test Dataset:")
print(test_df.isna().sum())

Standard Missing Values in the Train Dataset:
age               0
workclass         0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
label->50K        0
dtype: int64

Standard missing values in Test Dataset:
age               0
workclass         0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
label->50K        0
dtype: int64


In [25]:
#Check for `"?"` Missing-Value Markers - not automatically done by pandas methods. Needs to be checked explicitly
question_marks_train = (train_df == "?").sum()
question_marks_test = (test_df == "?").sum()

# Display only columns that contain "?".
print(" '?' values in Train Data:")
print(question_marks_train[question_marks_train > 0])

print("\n'?' values in Test Data:")
print(question_marks_test[question_marks_test > 0])


#Replace the ? with numpy NaN
train_df = train_df.replace("?",np.nan)
test_df = test_df.replace("?", np.nan)

#Checking if missing values are being identified
print("Missing Values after conversion in Train Data:")
print(train_df.isna().sum()[train_df.isna().sum()>0])

print("\n Missing values after conversion in Test Data:")
print(test_df.isna().sum()[test_df.isna().sum()>0])

 '?' values in Train Data:
workclass         1447
occupation        1454
native_country     458
dtype: int64

'?' values in Test Data:
workclass         389
occupation        389
native_country    125
dtype: int64
Missing Values after conversion in Train Data:
workclass         1447
occupation        1454
native_country     458
dtype: int64

 Missing values after conversion in Test Data:
workclass         389
occupation        389
native_country    125
dtype: int64


## 6. Inspect Class Distribution

The target variable is `label->50K`, where:

- `0` represents annual income less than or equal to \$50,000.
- `1` represents annual income greater than \$50,000.

Because we know that the dataset is imbalanced, the class
counts and proportions are examined before model training.

In [26]:
#Counting number of observations belonging to each target class.
class_counts = (
    train_df["label->50K"].value_counts()
    .sort_index()
)

#Counting proportions
class_proportions = (
    train_df["label->50K"]
    .value_counts(normalize=True)
    .sort_index()
)

print("Training Class Counts:")
print(class_counts)

print("\nTraining Class Proportions:")
print(class_proportions)


Training Class Counts:
label->50K
0    19778
1     6270
Name: count, dtype: int64

Training Class Proportions:
label->50K
0    0.759291
1    0.240709
Name: proportion, dtype: float64


## 7. Majority-Class Baseline

Before Logistic Regression is evaluated, a naive majority-class baseline
is established.

This baseline ignores all predictor variables and assigns the majority
class to every observation in the test set. It provides a reference
showing how much accuracy can be obtained without learning any
relationship between the features and the target.

The trained model should perform meaningfully better than this baseline.

In [27]:
# Extract the test target values.
y_test = test_df["label->50K"]

# Identify the majority class using the training set only.
# This avoids using information from the test set when establishing
# the baseline prediction rule.
majority_class = train_df["label->50K"].mode()[0]

# Predict the majority class for every observation in the test set.
baseline_predictions = np.full(
    shape=len(y_test),
    fill_value=majority_class
)

# Calculate the test accuracy of the naive baseline.
baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Majority class:", majority_class)
print(f"Majority-class baseline accuracy: {baseline_accuracy:.4f}")
print(f"Majority-class baseline accuracy: {baseline_accuracy * 100:.2f}%")
print("Baseline F1:", f1_score(y_test, baseline_predictions))          # 0, never predicts 1
print("Baseline ROC-AUC:", roc_auc_score(y_test, baseline_predictions))  # 0.5

Majority class: 0
Majority-class baseline accuracy: 0.7588
Majority-class baseline accuracy: 75.88%
Baseline F1: 0.0
Baseline ROC-AUC: 0.5
